# 07 · Productionisation — *from scratch*

**Phase goal:** ship the winner as a **single, portable file** with a clean
serving contract — and build a friendly `predict()` around it.

## One `Pipeline` object = no train/serve skew
The shipped model is **one** scikit-learn `Pipeline` = `TargetEncoder` + model.
The exact preprocessing fitted on the training data is re-applied automatically
at inference, so the encoding can never drift between training and serving.

In [1]:
# --- Self-contained setup: schema, cleaning, features, bands ---------------
# (In the real project these live in src/car_pricing/*. Here we write them out
#  so this notebook imports nothing from the project.)
import warnings
warnings.filterwarnings("ignore")   # hide library deprecation notices for clean output

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder

DATA_PATH     = "../data/raw/cars24-car-price-cleaned-new.csv.gz"
TARGET        = "selling_price"
TARGET_ENCODE = ["make", "model"]
NUMERIC       = ["km_driven", "mileage", "engine", "max_power", "age"]
FLAGS         = ["Individual", "Trustmark Dealer", "Diesel", "Electric", "LPG",
                 "Petrol", "Manual", "Seats_5", "Seats_Above_5"]
FEATURES      = TARGET_ENCODE + NUMERIC + FLAGS
BAND_LABELS   = ["Low", "Medium", "High"]
TEST_SIZE, RANDOM_STATE, CV_FOLDS = 0.2, 42, 3


def clean(df):
    df = df.rename(columns={"5": "Seats_5", ">5": "Seats_Above_5"})
    df = df.dropna(subset=[TARGET])
    df = df[df[TARGET] > 0].dropna()
    for c in TARGET_ENCODE:
        df[c] = df[c].astype(str).str.strip().str.upper()
    return df.reset_index(drop=True)


def build_preprocessor():
    """Target-encode make/model; pass numeric + flag columns through."""
    return ColumnTransformer([
        ("target_enc", TargetEncoder(target_type="continuous", random_state=RANDOM_STATE), TARGET_ENCODE),
        ("passthrough", "passthrough", NUMERIC + FLAGS),
    ], remainder="drop")


def band_edges(prices):
    edges = [float(np.quantile(prices, q)) for q in (0.0, 1/3, 2/3, 1.0)]
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + 0.01
    return [round(e, 2) for e in edges]


def price_to_band(prices, edges):
    prices = np.asarray(prices, dtype=float)
    idx = np.clip(np.digitize(prices, edges[1:-1]), 0, len(BAND_LABELS) - 1)
    return np.array(BAND_LABELS)[idx]


df = clean(pd.read_csv(DATA_PATH))
X = df[FEATURES].copy()
y = df[TARGET].copy()
print(f"{len(df):,} rows | {len(FEATURES)} features")

19,820 rows | 16 features


In [2]:
def model_zoo():
    """Name -> a fresh, unfitted regressor. Modest, comparable settings so the
    bake-off is a FAIR fight, not a tuning contest (we tune later, in nb 08)."""
    from sklearn.linear_model import Ridge
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

    zoo = {
        "Ridge (linear)": Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "Decision Tree":  DecisionTreeRegressor(max_depth=12, min_samples_leaf=8, random_state=RANDOM_STATE),
        "Random Forest":  RandomForestRegressor(n_estimators=200, max_depth=18,
                                                min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06,
                                                              max_depth=8, random_state=RANDOM_STATE),
    }
    # Add the gradient-boosting libraries only if they are installed.
    try:
        from xgboost import XGBRegressor
        zoo["XGBoost"] = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                      subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                                      random_state=RANDOM_STATE, verbosity=0)
    except ImportError:
        pass
    try:
        from lightgbm import LGBMRegressor
        zoo["LightGBM"] = LGBMRegressor(n_estimators=600, num_leaves=63, learning_rate=0.05,
                                        subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                                        random_state=RANDOM_STATE, verbosity=-1)
    except ImportError:
        pass
    return zoo


print("candidates:", list(model_zoo()))

candidates: ['Ridge (linear)', 'Decision Tree', 'Random Forest', 'HistGradientBoosting', 'XGBoost', 'LightGBM']


In [3]:
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Train the shippable pipeline (same recipe as nb 06).
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
winner_name = "LightGBM" if "LightGBM" in model_zoo() else "HistGradientBoosting"

pipe = Pipeline([
    ("preprocess", build_preprocessor()),
    ("model", model_zoo()[winner_name]),
])
pipe.fit(X_tr, y_tr)

# Persist ONE artifact. compress keeps the file small.
OUT_DIR = Path("../models"); OUT_DIR.mkdir(exist_ok=True)
ARTIFACT = OUT_DIR / "price_pipeline_basics.pkl"
joblib.dump(pipe, ARTIFACT, compress=("lzma", 9))

mb = ARTIFACT.stat().st_size / 1024 / 1024
print(f"saved {ARTIFACT.name}  ({mb:.2f} MB)")
print("steps:", [s for s, _ in pipe.steps])

saved price_pipeline_basics.pkl  (0.92 MB)
steps: ['preprocess', 'model']


### Serving metadata: how the API auto-fills what the caller omits
A friendly API lets a caller pass only `make` + `model` and fills the rest from
that model's typical values. We pre-compute those "typical values" once.

In [4]:
# Per-model median specs, plus dataset-wide fallbacks. This little bundle is
# everything the predict() below needs to auto-fill missing inputs.
edges = band_edges(y_tr)
spec_cols = ["mileage", "engine", "max_power"]

model_specs = {}   # make -> model -> {mileage, engine, max_power}
for (mk, mdl), row in df.groupby(["make", "model"])[spec_cols].median().round(1).iterrows():
    model_specs.setdefault(mk, {})[mdl] = {c: float(row[c]) for c in spec_cols}

numeric_defaults = {c: float(df[c].median()) for c in NUMERIC}
print("band edges   :", edges)
print("numeric dflts:", numeric_defaults)

band edges   : [0.3, 3.99, 6.75, 20.9]
numeric dflts: {'km_driven': 52000.0, 'mileage': 19.3, 'engine': 1248.0, 'max_power': 86.8, 'age': 8.0}


In [5]:
def format_price(lakhs):
    """Render a price (in Lakhs) in Indian Crore/Lakh/Rupee notation."""
    if lakhs >= 100:
        return f"Rs {lakhs / 100:.2f} Crore"
    if lakhs >= 1:
        return f"Rs {lakhs:.2f} Lakhs"
    return f"Rs {lakhs * 100_000:,.0f}"


# Maps a friendly choice (e.g. fuel='Diesel') to which 0/1 flag columns to set.
FLAG_MAP = {
    "seller":       {"Dealer": [], "Individual": ["Individual"], "Trustmark Dealer": ["Trustmark Dealer"]},
    "fuel":         {"CNG": [], "Petrol": ["Petrol"], "Diesel": ["Diesel"], "Electric": ["Electric"], "LPG": ["LPG"]},
    "transmission": {"Automatic": [], "Manual": ["Manual"]},
    "seats":        {"Fewer than 5": [], "5": ["Seats_5"], "More than 5": ["Seats_Above_5"]},
}


def predict(payload):
    """Turn a friendly car description into a price + budget band.

    Only make + model are required; anything else falls back to that model's
    typical specs, then to dataset-wide medians.
    """
    make  = str(payload["make"]).strip().upper()
    model = str(payload["model"]).strip().upper()
    specs = (model_specs.get(make, {}).get(model)
             or {k: numeric_defaults[k] for k in spec_cols})

    row = {c: 0 for c in FEATURES}                       # start all-zero
    row["make"], row["model"] = make, model
    row["age"]       = float(payload.get("age", numeric_defaults["age"]))
    row["km_driven"] = float(payload.get("km_driven", numeric_defaults["km_driven"]))
    for c in spec_cols:
        row[c] = float(payload.get(c, specs[c]))
    for group, default in (("seller", "Dealer"), ("fuel", "Petrol"),
                           ("transmission", "Manual"), ("seats", "5")):
        for flag in FLAG_MAP[group].get(str(payload.get(group, default)), []):
            row[flag] = 1

    X_one = pd.DataFrame([row])[FEATURES]
    price = max(float(pipe.predict(X_one)[0]), 0.0)
    band  = str(price_to_band([price], edges)[0])
    return {"predicted_price_lakhs": round(price, 2),
            "display": format_price(price), "price_band": band}


# Try it: give what you know, the rest is auto-filled.
for car in [{"make": "MARUTI", "model": "SWIFT VXI", "age": 5, "km_driven": 40000},
            {"make": "BMW", "model": "X5", "age": 4},
            {"make": "HYUNDAI", "model": "CRETA", "fuel": "Diesel", "transmission": "Automatic"}]:
    print(car, "->", predict(car))

{'make': 'MARUTI', 'model': 'SWIFT VXI', 'age': 5, 'km_driven': 40000} -> {'predicted_price_lakhs': 5.74, 'display': 'Rs 5.74 Lakhs', 'price_band': 'Medium'}
{'make': 'BMW', 'model': 'X5', 'age': 4} -> {'predicted_price_lakhs': 13.53, 'display': 'Rs 13.53 Lakhs', 'price_band': 'High'}
{'make': 'HYUNDAI', 'model': 'CRETA', 'fuel': 'Diesel', 'transmission': 'Automatic'} -> {'predicted_price_lakhs': 6.48, 'display': 'Rs 6.48 Lakhs', 'price_band': 'Medium'}


### How it deploys, and the MLOps loop
- **Batch:** `pipe.predict(df)` over a table — e.g. nightly repricing.
- **REST API:** wrap `predict()` in Flask/FastAPI (the sibling Flask app shows the
  pattern with Docker + AWS ECS).
- **Interactive:** a Streamlit UI (the sibling Streamlit app).

**The loop that keeps it healthy:** CI runs tests on every push → CT (continuous
training) retrains on new data, and the **KPI gate blocks a worse model from
shipping** → monitoring watches live error and input drift → retrain when either
slips. Notebook 09 builds each of those pieces from scratch.